In [2]:
conda install -c conda-forge cartopy


3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\isabe\miniconda3\envs\sds-env

  added / updated specs:
    - cartopy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    cartopy-0.25.0             |  py314hd8fd7ce_1         1.5 MB  conda-forge
    pyshp-3.0.3                |     pyhd8ed1ab_0         444 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         2.0 MB

The following NEW packages will be INSTALLED:

  cartopy            conda-forge/win-64::cartopy-0.25.0-py314hd8fd7ce_1 
  pyshp              conda-forge/noarch::pyshp-3.0.3-pyhd8ed1ab_0 



cartopy-0.25.0       | 1.5 MB    |            |   0% 

pyshp-3.0.3          | 444 KB    |            |   0% 

pyshp-3.0.3          | 444 



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda




In [3]:
conda install -c conda-forge geodatasets

3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\isabe\miniconda3\envs\sds-env

  added / updated specs:
    - geodatasets


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    geodatasets-2026.1.0       |     pyhd8ed1ab_0          25 KB  conda-forge
    pooch-1.9.0                |     pyhd8ed1ab_0          56 KB  conda-forge
    ------------------------------------------------------------
                                           Total:          81 KB

The following NEW packages will be INSTALLED:

  geodatasets        conda-forge/noarch::geodatasets-2026.1.0-pyhd8ed1ab_0 
  pooch              conda-forge/noarch::pooch-1.9.0-pyhd8ed1ab_0 



pooch-1.9.0          | 56 KB     |            |   0% 

geodatasets-2026.1.0 | 25 KB     |            |   0% 
pooch-1.9.0          



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda




In [ ]:
## project first draft ##
# this file is for experimenting, writing first bits of code and testing

# import important packages
import requests
import pandas as pd
import geopandas as gpd
import time
import matplotlib.pyplot as plt
import folium
import contextily
import cmcrameri


from cartopy import crs as ccrs
from geodatasets import get_path

c:\Users\isabe\miniconda3\envs\sds-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
conda install -c conda-forge ipywidgets

3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\isabe\miniconda3\envs\sds-env

  added / updated specs:
    - ipywidgets


The following NEW packages will be INSTALLED:

  ipywidgets         conda-forge/noarch::ipywidgets-8.1.8-pyhd8ed1ab_0 
  jupyterlab_widgets conda-forge/noarch::jupyterlab_widgets-3.0.16-pyhcf101f3_1 
  widgetsnbextension conda-forge/noarch::widgetsnbextension-4.0.15-pyhd8ed1ab_0 



Preparing transaction: done
Verifying transaction: done
Executing transaction: done

Note: you may need to restart the kernel to use updated packages.




==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda




In [ ]:
# test my MAP_KEY

map_key = "ea49e5fe6beaf3a00954c71727386596"

import pandas as pd
import requests
api_url = "https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?map_key=" + map_key
try:
  response = requests.get(api_url)

  if response.status_code == 200:
    data = response.json()
    df = pd.Series(data)
    display(df)
  else:
    print(f"Error in the query: HTTP {response.status_code}")
    print(response.text)

  
except Exception as e:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print (f"There is an issue with the query: {e}\n try in your browser: {api_url}")
  


transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object

In [10]:
#lumo

import pandas as pd
import requests
from datetime import date

# --- KONFIGURATION ---
API_KEY = "ea49e5fe6beaf3a00954c71727386596"  # Ersetze dies mit deinem gültigen Key
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/"

# 1. Datum festlegen (Heute)
# FIRMS benötigt das Format YYYY-MM-DD
today = date.today().strftime("%Y-%m-%d")

# 2. Region definieren (Beispiel: Ein Rechteck über Deutschland)
# Format: lat_min,lon_min,lat_max,lon_max
# Oder als GeoJSON Polygon-String
region_coords = "47.0,5.0,55.0,15.0"  # Beispiel: Südwest bis Nordost Deutschland

# 3. Sensor und Produkt wählen
# Sensoren: 'VIIRS', 'MODIS', 'SNPP'
# Produkte: 'fire' (Feuer), 'thermal_anomalies' (Thermische Anomalien)
sensor = "VIIRS"
product = "fire"

# --- API AUFRUF ---
params = {
    "key": API_KEY,
    "date": today,
    "area": region_coords,
    "sensor": sensor,
    "product": product,
    "format": "csv"  # FIRMS liefert oft CSV zurück, das ist einfacher zu parsen
}

print(f"Abfrage für Datum: {today}, Region: {region_coords}")

try:
    response = requests.get(BASE_URL, params=params)
    
    # Statuscode prüfen
    if response.status_code == 200:
        # Die API gibt oft CSV zurück, nicht JSON
        # Wir nutzen pandas, um das CSV direkt zu lesen
        df = pd.read_csv(pd.io.common.StringIO(response.text))
        
        print(f"\nErfolgreich! {len(df)} Feuerstellen gefunden.")
        display(df.head()) # Zeige die ersten 5 Zeilen
        
        # Optional: Speichern
        # df.to_csv("firms_data_today.csv", index=False)
        
    elif response.status_code == 401:
        print("Fehler 401: Ungültiger API-Key. Bitte überprüfe deinen Key.")
    elif response.status_code == 400:
        print("Fehler 400: Falsche Parameter. Prüfe Datum und Koordinaten.")
        print(f"URL: {response.url}")
        print(f"Inhalt: {response.text[:200]}")
    else:
        print(f"Fehler: Status {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"Ein unerwarteter Fehler trat auf: {e}")

Abfrage für Datum: 2026-05-07, Region: 47.0,5.0,55.0,15.0
Ein unerwarteter Fehler trat auf: Error tokenizing data. C error: Expected 1 fields in line 7, saw 13

